# 08 - Causal Production Decisioning (10/10 Gate)

This notebook is the final rollout gate for experiment-driven product decisions.

It integrates evidence from:

- **04** causal inference (A/B + DiD)
- **05** cohort/time-series diagnostics
- **06** deployment scoring context
- **07** production model readiness

and applies study-guide principles from chapters **3, 4, 5, 7, 8**.

In [1]:
# Cell 0: Setup + artifact paths
from __future__ import annotations

import importlib.util
import json
from datetime import UTC, datetime
from pathlib import Path
import sys
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

required_libs = ["numpy", "pandas"]
missing_libs = [lib for lib in required_libs if importlib.util.find_spec(lib) is None]
if missing_libs:
    raise RuntimeError(
        "Missing required packages for Notebook 08: "
        + ", ".join(missing_libs)
        + "; install with: apps/backend/.venv/bin/python -m pip install -r ml/requirements.txt"
    )

ROOT = Path.cwd()
for pth in [ROOT, *ROOT.parents]:
    if (pth / "ml" / "pipelines" / "causal_decision_gate.py").exists():
        ROOT = pth
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PATHS = {
    "AB_RESULTS_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "ab_test" / "ab_test_results.json",
    "DID_RESULTS_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "did" / "did_results.json",
    "CAUSAL_DECISION_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "causal_release_decision.json",
    "CAUSAL_DECISION_MD_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "causal_release_decision.md",
    "COHORT_SUMMARY_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "cohort_time_series" / "cohort_time_series_summary.json",
    "COHORT_FIXED_WINDOW_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "cohort_time_series" / "cohort_fixed_window_metrics.csv",
    "DEPLOY_SUMMARY_PATH": ROOT / "ml" / "data" / "reports" / "churn" / "deployment_scoring_summary.json",
    "MODEL_RELEASE_PATH": ROOT / "ml" / "models" / "release_preflight_report.json",
    "METRICS_PATH": ROOT / "ml" / "models" / "churn_reorder_metrics.json",
    "FEATURE_DATASET_PATH": ROOT / "ml" / "data" / "churn_training_features.csv",
    "CAUSAL_CHECKS_CSV_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "causal_production_checks.csv",
    "CAUSAL_STUDY_AUDIT_JSON_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "causal_study_guide_audit.json",
    "CAUSAL_STUDY_AUDIT_MD_PATH": ROOT / "ml" / "data" / "reports" / "causal" / "causal_study_guide_audit.md",
}

path_df = pd.DataFrame(
    [{"artifact": k, "exists": p.exists(), "path": str(p)} for k, p in PATHS.items()]
).sort_values(["exists", "artifact"], ascending=[True, True]).reset_index(drop=True)

print("ROOT:", ROOT)
display(path_df)

ROOT: /Users/deliorincon/Desktop/Sliceiq


,artifact,exists,path
0,CAUSAL_CHECKS_CSV_PATH,False,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
1,CAUSAL_DECISION_MD_PATH,False,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
2,CAUSAL_STUDY_AUDIT_JSON_PATH,False,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
3,CAUSAL_STUDY_AUDIT_MD_PATH,False,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
4,AB_RESULTS_PATH,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
5,CAUSAL_DECISION_PATH,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
6,COHORT_FIXED_WINDOW_PATH,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
7,COHORT_SUMMARY_PATH,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
8,DEPLOY_SUMMARY_PATH,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...
9,DID_RESULTS_PATH,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...


In [2]:
# Cell 1: Load prior notebook outputs
def _read_json(path: Path, default: Any):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default


def _read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()


def _safe_float(v: Any) -> float:
    try:
        x = float(v)
        return x if np.isfinite(x) else float("nan")
    except Exception:
        return float("nan")


ab = _read_json(PATHS["AB_RESULTS_PATH"], {})
did = _read_json(PATHS["DID_RESULTS_PATH"], {})
prior_causal_decision = _read_json(PATHS["CAUSAL_DECISION_PATH"], {})
cohort_summary = _read_json(PATHS["COHORT_SUMMARY_PATH"], {})
deploy_summary = _read_json(PATHS["DEPLOY_SUMMARY_PATH"], {})
model_release = _read_json(PATHS["MODEL_RELEASE_PATH"], {})
metrics = _read_json(PATHS["METRICS_PATH"], {})

cohort_fixed_window = _read_csv(PATHS["COHORT_FIXED_WINDOW_PATH"])
feature_dataset = _read_csv(PATHS["FEATURE_DATASET_PATH"])

alpha = _safe_float(prior_causal_decision.get("alpha"))
if pd.isna(alpha):
    alpha = 0.05

thresholds = prior_causal_decision.get("thresholds", {}) if isinstance(prior_causal_decision, dict) else {}
min_binary_lift = _safe_float(thresholds.get("practical_min_conversion_lift"))
if pd.isna(min_binary_lift):
    min_binary_lift = 0.005
min_revenue_lift = _safe_float(thresholds.get("practical_min_revenue_lift"))
if pd.isna(min_revenue_lift):
    min_revenue_lift = 0.0
min_did_effect = _safe_float(thresholds.get("practical_min_did_effect"))
if pd.isna(min_did_effect):
    min_did_effect = 0.0

context_rows = [
    {"source": "03_training", "signal": "model_version", "value": metrics.get("model_version")},
    {"source": "03_training", "signal": "test_roc_auc", "value": _safe_float(metrics.get("test_metrics", {}).get("roc_auc"))},
    {"source": "03_training", "signal": "test_pr_auc", "value": _safe_float(metrics.get("test_metrics", {}).get("pr_auc"))},
    {"source": "04_causal", "signal": "prior_final_decision", "value": prior_causal_decision.get("final_decision")},
    {"source": "05_cohort_ts", "signal": "daily_anomalies", "value": cohort_summary.get("daily_anomalies")},
    {"source": "06_deployment", "signal": "scored_users", "value": deploy_summary.get("scored_users")},
    {"source": "06_deployment", "signal": "high_risk_share", "value": deploy_summary.get("risk_mix", {}).get("high")},
    {"source": "07_prod_readiness", "signal": "model_release_decision", "value": model_release.get("final_decision")},
]
context_df = pd.DataFrame(context_rows)
display(context_df)

,source,signal,value
0,03_training,model_version,v20260311045627
1,03_training,test_roc_auc,0.998048
2,03_training,test_pr_auc,0.997594
3,04_causal,prior_final_decision,ship
4,05_cohort_ts,daily_anomalies,6
5,06_deployment,scored_users,200
6,06_deployment,high_risk_share,0.695
7,07_prod_readiness,model_release_decision,ship


## Unified Causal Gate Checks

This section validates:

- A/B design integrity (SRM, readiness, covariate balance, multiple testing)
- A/B lift quality (binary and continuous/CUPED)
- DiD quality (readiness, significance, practical effect, pretrend/placebo guardrails)
- Cross-notebook rollout context (cohort health, deployment mix, model release status)

In [3]:
# Cell 2: Build causal + study-guide checks table
checks: list[dict[str, Any]] = []


def _status(ok: bool, *, warn: bool = False) -> str:
    if ok:
        return "pass"
    return "warn" if warn else "fail"


def _status_max(value: float, pass_max: float, warn_max: float) -> str:
    if pd.isna(value):
        return "warn"
    if value <= pass_max:
        return "pass"
    if value <= warn_max:
        return "warn"
    return "fail"


def _add_check(
    *,
    domain: str,
    check: str,
    status: str,
    value: Any,
    threshold: Any,
    evidence: str,
    why_it_matters: str,
    chapter: str | None = None,
) -> None:
    checks.append(
        {
            "domain": domain,
            "chapter": chapter,
            "check": check,
            "status": status,
            "value": value,
            "threshold": threshold,
            "evidence": evidence,
            "why_it_matters": why_it_matters,
        }
    )


# Artifact existence
for key in [
    "AB_RESULTS_PATH",
    "DID_RESULTS_PATH",
    "COHORT_SUMMARY_PATH",
    "DEPLOY_SUMMARY_PATH",
    "MODEL_RELEASE_PATH",
    "METRICS_PATH",
]:
    p = PATHS[key]
    _add_check(
        domain="artifacts",
        chapter=None,
        check=f"{key.lower()}_exists",
        status=_status(p.exists()),
        value=p.exists(),
        threshold=True,
        evidence=str(p),
        why_it_matters="Causal production decisioning depends on upstream notebook artifacts.",
    )

# A/B checks (Chapter 7)
ab_readiness = ab.get("readiness", {}) if isinstance(ab, dict) else {}
ab_binary_ready = bool(ab_readiness.get("ab_binary_ready", False))
ab_cont_ready = bool(ab_readiness.get("ab_continuous_ready", False))
ab_hetero_ready = bool(ab_readiness.get("ab_heterogeneity_ready", False))

_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_binary_ready",
    status=_status(ab_binary_ready),
    value=ab_binary_ready,
    threshold=True,
    evidence="ab_test_results.json :: readiness.ab_binary_ready",
    why_it_matters="Binary conversion inference must meet minimum data support.",
)
_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_continuous_ready",
    status=_status(ab_cont_ready),
    value=ab_cont_ready,
    threshold=True,
    evidence="ab_test_results.json :: readiness.ab_continuous_ready",
    why_it_matters="Continuous-metric inference requires adequate per-variant support.",
)
_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_heterogeneity_ready",
    status=_status(ab_hetero_ready, warn=True),
    value=ab_hetero_ready,
    threshold=True,
    evidence="ab_test_results.json :: readiness.ab_heterogeneity_ready",
    why_it_matters="Segment-level heterogeneity is required for targeted rollout policy.",
)

srm_flag = bool(ab.get("srm", {}).get("flag", True)) if isinstance(ab, dict) else True
_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_srm_guardrail",
    status=_status(not srm_flag),
    value=srm_flag,
    threshold="False",
    evidence="ab_test_results.json :: srm.flag",
    why_it_matters="SRM failures undermine randomization validity.",
)

binary = ab.get("binary_results", {}) if isinstance(ab, dict) else {}
binary_p = _safe_float(binary.get("p_value"))
binary_lift = _safe_float(binary.get("rate_diff"))
binary_practical_ok = bool(binary.get("practical_ok", False))
ab_binary_effect_ok = pd.notna(binary_p) and (binary_p < alpha) and pd.notna(binary_lift) and (binary_lift >= min_binary_lift) and binary_practical_ok
_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_binary_effect_quality",
    status=_status(ab_binary_effect_ok, warn=True),
    value={"p_value": binary_p, "rate_diff": binary_lift, "practical_ok": binary_practical_ok},
    threshold=f"p<{alpha} and lift>={min_binary_lift}",
    evidence="ab_test_results.json :: binary_results",
    why_it_matters="Statistical and practical lift should both pass before broad rollout.",
)

cuped = ab.get("cuped_results", {}) if isinstance(ab, dict) else {}
cuped_p = _safe_float(cuped.get("welch_p_value"))
cuped_diff = _safe_float(cuped.get("difference"))
cuped_var_red = _safe_float(cuped.get("variance_reduction"))
ab_cuped_ok = pd.notna(cuped_p) and (cuped_p < alpha) and pd.notna(cuped_diff) and (cuped_diff >= min_revenue_lift)
_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_cuped_effect_quality",
    status=_status(ab_cuped_ok, warn=True),
    value={"p_value": cuped_p, "difference": cuped_diff, "variance_reduction": cuped_var_red},
    threshold=f"p<{alpha} and difference>={min_revenue_lift}",
    evidence="ab_test_results.json :: cuped_results",
    why_it_matters="CUPED-adjusted effects improve precision for rollout confidence.",
)

smd_vals = []
for row in ab.get("covariate_balance_smd", []):
    if isinstance(row, dict):
        smd_vals.append(abs(_safe_float(row.get("smd_treat_minus_control"))))
max_abs_smd = max([v for v in smd_vals if pd.notna(v)], default=np.nan)
_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_covariate_balance_smd",
    status=_status_max(max_abs_smd, pass_max=0.10, warn_max=0.20),
    value=max_abs_smd,
    threshold="<= 0.10 (warn <= 0.20)",
    evidence="ab_test_results.json :: covariate_balance_smd",
    why_it_matters="Large imbalance can bias treatment effect estimates.",
)

risk_bin_tests = ab.get("risk_band_binary_tests", []) if isinstance(ab, dict) else []
risk_cont_tests = ab.get("risk_band_continuous_tests", []) if isinstance(ab, dict) else []
fdr_present = bool(risk_bin_tests) and all(("q_value_bh" in r) for r in risk_bin_tests if isinstance(r, dict))
heterogeneity_coverage_ok = len(risk_bin_tests) >= 2 and len(risk_cont_tests) >= 2

_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_multiple_testing_control",
    status=_status(fdr_present, warn=True),
    value=fdr_present,
    threshold=True,
    evidence="ab_test_results.json :: risk_band_*_tests with q_value_bh",
    why_it_matters="Chapter 7 recommends explicit control for segmented testing multiplicity.",
)
_add_check(
    domain="causal_ab",
    chapter="ch7",
    check="ab_heterogeneity_coverage",
    status=_status(heterogeneity_coverage_ok, warn=True),
    value={"binary_bands": len(risk_bin_tests), "continuous_bands": len(risk_cont_tests)},
    threshold=">= 2 risk bands per metric",
    evidence="ab_test_results.json :: risk_band tests",
    why_it_matters="Consistent uplift across segments supports stable production targeting.",
)

# DiD checks (Chapter 7 + Chapter 3 period integrity)
did_readiness = did.get("readiness", {}) if isinstance(did, dict) else {}
did_ready_reg = bool(did_readiness.get("did_ready_for_regression", False))
_add_check(
    domain="causal_did",
    chapter="ch7",
    check="did_ready_for_regression",
    status=_status(did_ready_reg),
    value=did_ready_reg,
    threshold=True,
    evidence="did_results.json :: readiness.did_ready_for_regression",
    why_it_matters="Regression DiD should only run when support and cells are sufficient.",
)

base_checks = prior_causal_decision.get("checks", {}) if isinstance(prior_causal_decision, dict) else {}
did_stat_ok = bool(base_checks.get("did_stat_ok", False))
did_practical_ok = bool(base_checks.get("did_practical_ok", False))
pretrend_ok = bool(base_checks.get("pretrend_ok", False))
placebo_ok = bool(base_checks.get("placebo_ok", False))

_add_check(
    domain="causal_did",
    chapter="ch7",
    check="did_primary_effect_significance",
    status=_status(did_stat_ok, warn=True),
    value=did_stat_ok,
    threshold=True,
    evidence="causal_release_decision.json :: checks.did_stat_ok",
    why_it_matters="Significance is a minimum requirement for causal rollout confidence.",
)
_add_check(
    domain="causal_did",
    chapter="ch7",
    check="did_primary_effect_practical",
    status=_status(did_practical_ok, warn=True),
    value=did_practical_ok,
    threshold=True,
    evidence="causal_release_decision.json :: checks.did_practical_ok",
    why_it_matters="Practical effect size determines whether rollout impact is meaningful.",
)
_add_check(
    domain="causal_did",
    chapter="ch7",
    check="did_parallel_trends_guardrail",
    status=_status(pretrend_ok, warn=True),
    value=pretrend_ok,
    threshold=True,
    evidence="causal_release_decision.json :: checks.pretrend_ok",
    why_it_matters="Parallel trends is central to DiD identification validity.",
)
_add_check(
    domain="causal_did",
    chapter="ch7",
    check="did_placebo_guardrail",
    status=_status(placebo_ok, warn=True),
    value=placebo_ok,
    threshold=True,
    evidence="causal_release_decision.json :: checks.placebo_ok",
    why_it_matters="Placebo checks reduce risk of spurious treatment effects.",
)

did_manual = did.get("manual", {}) if isinstance(did, dict) else {}
did_manual_complete = all(
    pd.notna(_safe_float(did_manual.get(k)))
    for k in ["control_pre", "control_post", "treatment_pre", "treatment_post", "did_manual"]
)
_add_check(
    domain="causal_did",
    chapter="ch3",
    check="did_pre_post_period_integrity",
    status=_status(did_manual_complete, warn=True),
    value=did_manual_complete,
    threshold=True,
    evidence="did_results.json :: manual pre/post cells",
    why_it_matters="Chapter 3 time comparisons require complete and consistent period definitions.",
)

# Cohort + time-series context (Chapter 4 + 3)
cohort_count = int(cohort_summary.get("cohorts", -1)) if isinstance(cohort_summary, dict) else -1
max_period = int(cohort_summary.get("max_period", -1)) if isinstance(cohort_summary, dict) else -1
daily_anomalies = int(cohort_summary.get("daily_anomalies", -1)) if isinstance(cohort_summary, dict) else -1

_add_check(
    domain="context",
    chapter="ch4",
    check="cohort_coverage",
    status=_status(cohort_count >= 3),
    value=cohort_count,
    threshold=">= 3 cohorts",
    evidence="cohort_time_series_summary.json :: cohorts",
    why_it_matters="Cohort breadth improves external validity of observed treatment behavior.",
)
_add_check(
    domain="context",
    chapter="ch3",
    check="cohort_period_depth",
    status=_status(max_period >= 4, warn=True),
    value=max_period,
    threshold=">= 4 periods",
    evidence="cohort_time_series_summary.json :: max_period",
    why_it_matters="Adequate period depth supports stable time-based interpretation.",
)
_add_check(
    domain="context",
    chapter="ch3",
    check="daily_anomaly_pressure",
    status=_status_max(float(daily_anomalies), pass_max=12.0, warn_max=20.0) if daily_anomalies >= 0 else "warn",
    value=daily_anomalies,
    threshold="<= 12 (warn <= 20)",
    evidence="cohort_time_series_summary.json :: daily_anomalies",
    why_it_matters="High anomaly pressure can invalidate near-term rollout assumptions.",
)

if not cohort_fixed_window.empty and "avg_orders_90d" in cohort_fixed_window.columns:
    fixed_window_coverage = float(pd.to_numeric(cohort_fixed_window["avg_orders_90d"], errors="coerce").notna().mean())
else:
    fixed_window_coverage = np.nan
_add_check(
    domain="context",
    chapter="ch4",
    check="fixed_window_fairness_coverage",
    status=_status(pd.notna(fixed_window_coverage) and fixed_window_coverage >= 0.40, warn=True),
    value=fixed_window_coverage,
    threshold=">= 0.40 cohorts with mature 90d window",
    evidence="cohort_fixed_window_metrics.csv",
    why_it_matters="Chapter 4 fairness requires fixed windows to avoid exposure bias.",
)

# Cross-notebook production context
model_gate_decision = str(model_release.get("final_decision", "unknown")) if isinstance(model_release, dict) else "unknown"
model_gate_pass = model_gate_decision == "ship"
_add_check(
    domain="cross_notebook",
    chapter="ch8",
    check="model_production_gate_status",
    status=_status(model_gate_pass, warn=True),
    value=model_gate_decision,
    threshold="ship",
    evidence="release_preflight_report.json :: final_decision",
    why_it_matters="Causal rollout should align with model-system readiness state.",
)

scored_users = int(deploy_summary.get("scored_users", -1)) if isinstance(deploy_summary, dict) else -1
_add_check(
    domain="cross_notebook",
    chapter="ch8",
    check="deployment_sample_size",
    status=_status(scored_users >= 100),
    value=scored_users,
    threshold=">= 100",
    evidence="deployment_scoring_summary.json :: scored_users",
    why_it_matters="Tiny live populations reduce confidence in rollout generalization.",
)

# External validity: compare A/B band mix to current live risk mix.
ab_band_share = {}
if risk_bin_tests:
    total = 0.0
    for row in risk_bin_tests:
        if not isinstance(row, dict):
            continue
        n_c = _safe_float(row.get("n_control"))
        n_t = _safe_float(row.get("n_treatment"))
        n = (0.0 if pd.isna(n_c) else n_c) + (0.0 if pd.isna(n_t) else n_t)
        band = str(row.get("risk_band", "unknown"))
        ab_band_share[band] = ab_band_share.get(band, 0.0) + n
        total += n
    if total > 0:
        ab_band_share = {k: float(v / total) for k, v in ab_band_share.items()}

live_high_share = _safe_float(deploy_summary.get("risk_mix", {}).get("high")) if isinstance(deploy_summary, dict) else np.nan
ab_high_share = _safe_float(ab_band_share.get("high")) if ab_band_share else np.nan
mix_shift = abs(live_high_share - ab_high_share) if pd.notna(live_high_share) and pd.notna(ab_high_share) else np.nan
_add_check(
    domain="cross_notebook",
    chapter="ch8",
    check="ab_to_live_population_shift_high_band",
    status=_status_max(mix_shift, pass_max=0.40, warn_max=0.55),
    value=mix_shift,
    threshold="<= 0.40 (warn <= 0.55)",
    evidence=f"ab_high_share={ab_high_share}, live_high_share={live_high_share}",
    why_it_matters="Chapter 8 emphasizes production datasets may differ from experiment cohorts.",
)

# Chapter 5 text discipline
feature_cols = set(feature_dataset.columns) if not feature_dataset.empty else set()
text_proxies = {"avg_rating_lifetime", "review_count_lifetime"}
text_proxy_ok = text_proxies.issubset(feature_cols)
_add_check(
    domain="study_guide",
    chapter="ch5",
    check="text_proxy_signals_available",
    status=_status(text_proxy_ok, warn=True),
    value=text_proxy_ok,
    threshold=True,
    evidence=f"found={sorted(text_proxies.intersection(feature_cols))}",
    why_it_matters="Chapter 5 encourages preserving normalized text-derived signal where available.",
)

control_label = str(ab.get("control_label", "")).strip()
treatment_label = str(ab.get("treatment_label", "")).strip()
labels_normalized = bool(control_label and treatment_label and control_label == control_label.lower() and treatment_label == treatment_label.lower() and control_label != treatment_label)
_add_check(
    domain="study_guide",
    chapter="ch5",
    check="variant_label_normalization",
    status=_status(labels_normalized, warn=True),
    value={"control": control_label, "treatment": treatment_label},
    threshold="distinct lower-case labels",
    evidence="ab_test_results.json :: control_label/treatment_label",
    why_it_matters="Text normalization avoids silent group-key fragmentation.",
)

# Chapter 8 governance + privacy checks
governance_ok = (
    PATHS["AB_RESULTS_PATH"].exists()
    and PATHS["DID_RESULTS_PATH"].exists()
    and PATHS["MODEL_RELEASE_PATH"].exists()
    and PATHS["DEPLOY_SUMMARY_PATH"].exists()
)
_add_check(
    domain="study_guide",
    chapter="ch8",
    check="governance_artifact_bundle",
    status=_status(governance_ok),
    value=governance_ok,
    threshold=True,
    evidence="AB + DiD + model release + deployment summary",
    why_it_matters="Chapter 8 favors reusable, audited data products over ad-hoc query outputs.",
)

def _collect_keys(obj: Any, prefix: str = "") -> list[str]:
    keys: list[str] = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            keys.append(key)
            keys.extend(_collect_keys(v, key))
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:10]):
            keys.extend(_collect_keys(v, f"{prefix}[{i}]" if prefix else f"[{i}]"))
    return keys

pii_tokens = ("email", "phone", "address", "name")
pii_like_keys = [
    k for k in (_collect_keys(ab) + _collect_keys(did))
    if any(tok in k.lower() for tok in pii_tokens)
]
_add_check(
    domain="study_guide",
    chapter="ch8",
    check="causal_output_pii_hygiene",
    status=_status(len(pii_like_keys) == 0),
    value=len(pii_like_keys),
    threshold="0 pii-like keys",
    evidence=f"pii_like_keys_sample={pii_like_keys[:5]}",
    why_it_matters="Chapter 8 governance includes minimizing PII propagation in analytics artifacts.",
)

checks_df = pd.DataFrame(checks)
if checks_df.empty:
    raise RuntimeError("No checks generated in notebook 08")

status_order = {"fail": 0, "warn": 1, "pass": 2}
checks_df["status_rank"] = checks_df["status"].map(status_order).fillna(3).astype(int)
checks_df = checks_df.sort_values(["status_rank", "domain", "chapter", "check"]).reset_index(drop=True)

summary = checks_df["status"].value_counts().reindex(["pass", "warn", "fail"], fill_value=0)
study_df = checks_df[checks_df["chapter"].notna()].copy()
study_summary = study_df["status"].value_counts().reindex(["pass", "warn", "fail"], fill_value=0)

print("Causal check summary:", summary.to_dict())
print("Study-guide summary:", study_summary.to_dict())
display(checks_df)

Causal check summary: {'pass': 32, 'warn': 0, 'fail': 0}
Study-guide summary: {'pass': 26, 'warn': 0, 'fail': 0}


,domain,chapter,check,status,value,threshold,evidence,why_it_matters,status_rank
0,artifacts,NaN,ab_results_path_exists,pass,True,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,Causal production decisioning depends on upstr...,2
1,artifacts,NaN,cohort_summary_path_exists,pass,True,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,Causal production decisioning depends on upstr...,2
2,artifacts,NaN,deploy_summary_path_exists,pass,True,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,Causal production decisioning depends on upstr...,2
3,artifacts,NaN,did_results_path_exists,pass,True,True,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,Causal production decisioning depends on upstr...,2
4,artifacts,NaN,metrics_path_exists,pass,True,True,/Users/deliorincon/Desktop/Sliceiq/ml/models/c...,Causal production decisioning depends on upstr...,2
5,artifacts,NaN,model_release_path_exists,pass,True,True,/Users/deliorincon/Desktop/Sliceiq/ml/models/r...,Causal production decisioning depends on upstr...,2
6,causal_ab,ch7,ab_binary_effect_quality,pass,"{'p_value': 9.857681621527965e-12, 'rate_diff'...",p<0.05 and lift>=0.005,ab_test_results.json :: binary_results,Statistical and practical lift should both pas...,2
7,causal_ab,ch7,ab_binary_ready,pass,True,True,ab_test_results.json :: readiness.ab_binary_ready,Binary conversion inference must meet minimum ...,2
8,causal_ab,ch7,ab_continuous_ready,pass,True,True,ab_test_results.json :: readiness.ab_continuou...,Continuous-metric inference requires adequate ...,2
9,causal_ab,ch7,ab_covariate_balance_smd,pass,0.031024,<= 0.10 (warn <= 0.20),ab_test_results.json :: covariate_balance_smd,Large imbalance can bias treatment effect esti...,2


## Decision Engine

Decision policy:

- `iterate`: one or more critical failures.
- `pilot`: no failures, but warnings present.
- `ship`: clean checks and no warnings.

In [4]:
# Cell 3: Build final causal rollout decision
fail_checks = checks_df[checks_df["status"] == "fail"].copy()
warn_checks = checks_df[checks_df["status"] == "warn"].copy()

reason_codes = [f"FAIL_{c.upper()}" for c in fail_checks["check"].tolist()]
warning_codes = [f"WARN_{c.upper()}" for c in warn_checks["check"].tolist()]

if reason_codes:
    final_decision = "iterate"
elif warning_codes:
    final_decision = "pilot"
else:
    final_decision = "ship"

pass_count = int((checks_df["status"] == "pass").sum())
warn_count = int((checks_df["status"] == "warn").sum())
fail_count = int((checks_df["status"] == "fail").sum())
total_count = int(len(checks_df))

readiness_score = float((pass_count + 0.5 * warn_count) / max(total_count, 1))
study_total = int(len(study_df))
study_pass = int((study_df["status"] == "pass").sum())
study_warn = int((study_df["status"] == "warn").sum())
study_score = float((study_pass + 0.5 * study_warn) / max(study_total, 1))

# Core causal check booleans (backward-friendly fields)
core_checks = {
    "srm_flag": bool(srm_flag),
    "ab_binary_ready": bool(ab_binary_ready),
    "ab_continuous_ready": bool(ab_cont_ready),
    "did_ready_for_regression": bool(did_ready_reg),
    "ab_binary_stat_ok": bool(pd.notna(binary_p) and binary_p < alpha),
    "ab_binary_practical_ok": bool(pd.notna(binary_lift) and binary_lift >= min_binary_lift),
    "ab_revenue_stat_ok": bool(pd.notna(cuped_p) and cuped_p < alpha),
    "ab_revenue_practical_ok": bool(pd.notna(cuped_diff) and cuped_diff >= min_revenue_lift),
    "did_stat_ok": bool(did_stat_ok),
    "did_practical_ok": bool(did_practical_ok),
    "pretrend_ok": bool(pretrend_ok),
    "placebo_ok": bool(placebo_ok),
    "primary_did_model": base_checks.get("primary_did_model", "unknown"),
    "primary_did_coef": base_checks.get("primary_did_coef"),
    "primary_did_p": base_checks.get("primary_did_p"),
}

action_plan = []
if fail_count > 0:
    action_plan.append("Fix failed checks before rollout; rerun notebooks 04 -> 08.")
    if any(c in reason_codes for c in ["FAIL_AB_SRM_GUARDRAIL", "FAIL_AB_COVARIATE_BALANCE_SMD"]):
        action_plan.append("Repair experiment randomization / assignment quality and rerun A/B analysis.")
    if any(c in reason_codes for c in ["FAIL_DID_PARALLEL_TRENDS_GUARDRAIL", "FAIL_DID_PLACEBO_GUARDRAIL"]):
        action_plan.append("Re-specify DiD design (time window or controls) and retest assumptions.")
elif warn_count > 0:
    action_plan.append("Launch controlled pilot rollout with enhanced monitoring and predefined stop conditions.")
else:
    action_plan.append("Approve full rollout with standard weekly causal KPI monitoring.")

release_payload = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "final_decision": final_decision,
    "reason_codes": sorted(set(reason_codes)),
    "warning_codes": sorted(set(warning_codes)),
    "alpha": alpha,
    "thresholds": {
        "practical_min_conversion_lift": min_binary_lift,
        "practical_min_revenue_lift": min_revenue_lift,
        "practical_min_did_effect": min_did_effect,
    },
    "check_counts": {
        "pass": pass_count,
        "warn": warn_count,
        "fail": fail_count,
        "total": total_count,
    },
    "readiness_score": readiness_score,
    "study_guide_score": study_score,
    "checks": core_checks,
    "checks_table": checks_df.drop(columns=["status_rank"], errors="ignore").to_dict(orient="records"),
    "study_guide_summary": study_summary.to_dict(),
    "context": {
        "model_release_decision": model_release.get("final_decision"),
        "deployment_scored_users": deploy_summary.get("scored_users"),
        "deployment_high_risk_share": deploy_summary.get("risk_mix", {}).get("high"),
        "cohort_daily_anomalies": cohort_summary.get("daily_anomalies"),
        "prior_causal_decision": prior_causal_decision.get("final_decision"),
    },
    "action_plan": action_plan,
}

decision_df = pd.DataFrame(
    [
        {
            "final_decision": final_decision,
            "readiness_score": round(readiness_score, 4),
            "study_guide_score": round(study_score, 4),
            "fail_checks": fail_count,
            "warn_checks": warn_count,
            "reason_codes": ", ".join(release_payload["reason_codes"]) if release_payload["reason_codes"] else "NONE",
            "warning_codes": ", ".join(release_payload["warning_codes"]) if release_payload["warning_codes"] else "NONE",
        }
    ]
)

display(decision_df)
if not fail_checks.empty:
    display(fail_checks)
if not warn_checks.empty:
    display(warn_checks)

,final_decision,readiness_score,study_guide_score,fail_checks,warn_checks,reason_codes,warning_codes
0,ship,1.0,1.0,0,0,NONE,NONE


In [5]:
# Cell 4: Persist causal production decisioning artifacts
for key in [
    "CAUSAL_DECISION_PATH",
    "CAUSAL_DECISION_MD_PATH",
    "CAUSAL_CHECKS_CSV_PATH",
    "CAUSAL_STUDY_AUDIT_JSON_PATH",
    "CAUSAL_STUDY_AUDIT_MD_PATH",
]:
    PATHS[key].parent.mkdir(parents=True, exist_ok=True)

# Canonical decision output (used across project)
PATHS["CAUSAL_DECISION_PATH"].write_text(json.dumps(release_payload, indent=2), encoding="utf-8")

decision_md_lines = [
    "# Causal Production Release Decision",
    "",
    f"- Generated at (UTC): {release_payload['generated_at_utc']}",
    f"- Final decision: {release_payload['final_decision']}",
    f"- Readiness score: {release_payload['readiness_score']:.4f}",
    f"- Study guide score: {release_payload['study_guide_score']:.4f}",
    f"- Fail checks: {release_payload['check_counts']['fail']}",
    f"- Warn checks: {release_payload['check_counts']['warn']}",
    f"- Reason codes: {', '.join(release_payload['reason_codes']) if release_payload['reason_codes'] else 'NONE'}",
    f"- Warning codes: {', '.join(release_payload['warning_codes']) if release_payload['warning_codes'] else 'NONE'}",
    "",
    "## Core Check Signals",
    f"- SRM flag: {release_payload['checks']['srm_flag']}",
    f"- A/B binary ready: {release_payload['checks']['ab_binary_ready']}",
    f"- A/B continuous ready: {release_payload['checks']['ab_continuous_ready']}",
    f"- DiD ready: {release_payload['checks']['did_ready_for_regression']}",
    f"- Pretrend OK: {release_payload['checks']['pretrend_ok']}",
    f"- Placebo OK: {release_payload['checks']['placebo_ok']}",
    "",
    "## Action Plan",
]
for step in release_payload["action_plan"]:
    decision_md_lines.append(f"- {step}")
PATHS["CAUSAL_DECISION_MD_PATH"].write_text("\n".join(decision_md_lines), encoding="utf-8")

checks_df.drop(columns=["status_rank"], errors="ignore").to_csv(PATHS["CAUSAL_CHECKS_CSV_PATH"], index=False)

chapter_summary_df = (
    study_df.groupby(["chapter", "status"]).size().unstack(fill_value=0)
    if not study_df.empty
    else pd.DataFrame()
)
study_payload = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "summary": study_summary.to_dict(),
    "by_chapter": chapter_summary_df.to_dict(orient="index") if not chapter_summary_df.empty else {},
    "rows": study_df.drop(columns=["status_rank"], errors="ignore").to_dict(orient="records"),
}
PATHS["CAUSAL_STUDY_AUDIT_JSON_PATH"].write_text(json.dumps(study_payload, indent=2), encoding="utf-8")

study_md_lines = [
    "# Causal Study Guide Implementation Audit",
    "",
    f"- Generated at (UTC): {study_payload['generated_at_utc']}",
    f"- Pass: {study_payload['summary'].get('pass', 0)}",
    f"- Warn: {study_payload['summary'].get('warn', 0)}",
    f"- Fail: {study_payload['summary'].get('fail', 0)}",
    "",
    "## Detailed Checks",
]
for row in study_payload["rows"]:
    study_md_lines.append(
        f"- [{row['status'].upper()}] {row['chapter']} :: {row['check']} :: value={row['value']} :: threshold={row['threshold']}"
    )
PATHS["CAUSAL_STUDY_AUDIT_MD_PATH"].write_text("\n".join(study_md_lines), encoding="utf-8")

output_df = pd.DataFrame(
    [
        {"artifact": "causal_release_json", "path": str(PATHS["CAUSAL_DECISION_PATH"]), "exists": PATHS["CAUSAL_DECISION_PATH"].exists()},
        {"artifact": "causal_release_md", "path": str(PATHS["CAUSAL_DECISION_MD_PATH"]), "exists": PATHS["CAUSAL_DECISION_MD_PATH"].exists()},
        {"artifact": "causal_checks_csv", "path": str(PATHS["CAUSAL_CHECKS_CSV_PATH"]), "exists": PATHS["CAUSAL_CHECKS_CSV_PATH"].exists()},
        {"artifact": "causal_study_audit_json", "path": str(PATHS["CAUSAL_STUDY_AUDIT_JSON_PATH"]), "exists": PATHS["CAUSAL_STUDY_AUDIT_JSON_PATH"].exists()},
        {"artifact": "causal_study_audit_md", "path": str(PATHS["CAUSAL_STUDY_AUDIT_MD_PATH"]), "exists": PATHS["CAUSAL_STUDY_AUDIT_MD_PATH"].exists()},
    ]
)
display(output_df)
print("Final decision:", release_payload["final_decision"])


,artifact,path,exists
0,causal_release_json,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,True
1,causal_release_md,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,True
2,causal_checks_csv,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,True
3,causal_study_audit_json,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,True
4,causal_study_audit_md,/Users/deliorincon/Desktop/Sliceiq/ml/data/rep...,True


Final decision: ship


## Outcome

Use `causal_release_decision.json` as the canonical causal rollout decision artifact.